<a href="https://colab.research.google.com/github/snehilsarkar97/PredictiveAnalysis_RobotRace/blob/main/BaseModel.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ==============================================================
# 150-Day Wallet Simulation (Single Map: "Weapons Factory")
# --------------------------------------------------------------
# OUTPUT (printed at the end):
#   A small table with final wallet, level, and XP for each archetype.
# ==============================================================

from google.colab import drive
import pandas as pd
import json
import random
from bisect import bisect_right

# -----------------------------
# 1) LOAD REMOTE CONFIG (CSV)
# -----------------------------

# Mount Drive so we can read the CSV file.
drive.mount('/content/drive')

# Path to your RemoteConfig CSV in Drive (update if moved).
CONFIG_PATH = "/content/drive/MyDrive/OPT Volunteer Job Personal Folder/Week 5/remote config.csv"

# Read CSV and parse into a nested dict called `config`.
# Each row in the CSV has: key | type | TITLE | value
#  - If type == "json": value is a JSON string → parse to dict/list
#  - If type == "int" : parse to int
#  - Else             : keep as string
df = pd.read_csv(CONFIG_PATH)
config = {}
for _, row in df.iterrows():
    k, t, v = row["key"], row["type"], row["value"]
    if t == "json":
        config[k] = json.loads(v)
    elif t == "int":
        config[k] = int(v)
    else:
        config[k] = v

# -----------------------------
# 2) READ KNOBS FROM CONFIG
# -----------------------------
# RemoteConfig rows used (key → TITLE in your sheet):
# - IAPstore        → TITLE: "levelconfig"   (uses NewPlayerBonus)
# - LevelConfig     → TITLE: "levelconfig"   (uses level_config[])
# - GoldRewards     → TITLE: "levelconfig"   (uses gold_rewards[])
# - WageringSystem  → TITLE: "" (blank)      (uses maps.levelMilitary.{buy_in,rewards})

# Starting wallet for every archetype (from IAP store).
initial_wallet = config["IAPstore"]["NewPlayerBonus"]      # e.g., 1200

# Daily login bonus (fixed by you; not pulled from CSV on purpose).
daily_bonus_gold = 150

# Whether to credit one-time gold when a player crosses a new level.
use_levelup_gold = True

# Simulation length in days.
days = 150

# Number of racers in each race (your focal racer + opponents).
NUM_PLAYERS = 6

# The three archetypes we track in this simulation.
player_archetypes = ["Skilled", "Average", "Novice"]

# -----------------------------
# 3) RACE CADENCE
# -----------------------------
def races_today() -> int:
    """
    Return how many races run today.
    We run 3 every day, with a 50% chance of a 4th.
    """
    return 3 + (1 if random.random() < 0.5 else 0)

# -----------------------------
# 4) PERFORMANCE MODEL
# -----------------------------
# Gaussian skill means by archetype: higher mean → better average performance.
coupled_perf_mu = {"Skilled": 1.2, "Average": 1.0, "Novice": 0.6}

# Shared standard deviation for randomness in a race outcome.
coupled_perf_sigma = 0.35

# Opponent pool composition: probability weights to sample opponent archetypes.
opponent_mix_weights = {"Skilled": 0.25, "Average": 0.50, "Novice": 0.25}

# -----------------------------
# 5) PROGRESSION (XP & LEVELS)
# -----------------------------
# XP earned per race (simple constant).
xp_per_game_avg = 81.7

# Cumulative XP thresholds per level (from RemoteConfig → LevelConfig.level_config).
cum_xp_by_level = config["LevelConfig"]["level_config"]
cum_list = cum_xp_by_level[:]          # local copy used for bisect
max_level = len(cum_list)

# Convert total XP → current level using thresholds
def level_from_xp(xp_total: float) -> int:
    """
    Map cumulative XP to level by bisecting the threshold list.
    Levels are clamped to [1 .. max_level].
    """
    idx = bisect_right(cum_list, xp_total) - 1
    return max(1, min(idx + 1, max_level))

# One-time level-up gold: map level → gold, loaded from RemoteConfig → GoldRewards.gold_rewards
level_to_gold = {i + 1: g for i, g in enumerate(config["GoldRewards"]["gold_rewards"])}

# ---------------------------------------
# 6) MAP DATA (ONLY "WEAPONS FACTORY")
# ---------------------------------------
# Friendly names for map keys in config (for readability).
name_map = {
    "levelMilitary": "Weapons Factory",
    "levelLA": "Lost Angeles",
    "levelElectric": "Electric Downtown",
    "levelMushroom": "Biodome",
    "levelWhitecity": "White City",
    "levelChinese": "Chinese",
}

# WageringSystem.maps holds buy-in and post-fee rewards per map.
maps_cfg = config["WageringSystem"]["maps"]

# Build lookups: friendly name → rewards[], friendly name → buy_in
MAP_REWARDS = {name_map[k]: v["rewards"] for k, v in maps_cfg.items()}
MAP_BUYIN   = {name_map[k]: v["buy_in"]  for k, v in maps_cfg.items()}

# payout_from_rewards  #payotRAtion as per the player rank 1 2 3
def payout_from_rewards(map_name: str, place: int) -> int:
    """
    Return the post-fee prize from MAP_REWARDS based on finishing place.
    rewards[0] = 1st, rewards[1] = 2nd, rewards[2] = 3rd; all others get 0.
    """
    rewards = MAP_REWARDS[map_name]
    return rewards[place - 1] if 1 <= place <= 3 else 0

# ---------------------------------------
# 7) RACE OUTCOME (ONE FOCAL RACER)
# ---------------------------------------
def placement_coupled(focal_archetype: str):
    """
    Simulate one race for a single focal racer vs (NUM_PLAYERS-1) opponents.
    Steps:
      1) Sample opponent archetypes by opponent_mix_weights.
      2) Draw Gaussian performance for focal + opponents.
      3) Rank all racers by score and return focal's place: 1..3 or "other".
    """
    # Sample the opponent archetypes.
    kinds, weights = zip(*opponent_mix_weights.items())
    opponents = [random.choices(kinds, weights=weights, k=1)[0] for _ in range(NUM_PLAYERS - 1)]

    # Draw performance scores.
    focal_score = random.gauss(coupled_perf_mu[focal_archetype], coupled_perf_sigma)
    opp_scores  = [random.gauss(coupled_perf_mu[a], coupled_perf_sigma) for a in opponents]
    scores      = [focal_score] + opp_scores

    # Compute finishing place for the focal racer (index 0).
    order = sorted(range(NUM_PLAYERS), key=lambda i: scores[i], reverse=True)
    rank = order.index(0) + 1
    return rank if rank <= 3 else "other"

# -----------------------------
# 8) SIMULATION STATE
# -----------------------------
# Current gold, total XP, and level per archetype.
wallets = {n: initial_wallet for n in player_archetypes}
xp_tot  = {n: 0.0 for n in player_archetypes}
levels  = {n: 1   for n in player_archetypes}

# CURRENT MAP SELECTION IS **HARDCODED HERE** (as requested):
# We always race on "Weapons Factory" for the entire sim.
fixed_map_name = "Weapons Factory"
buy_in = MAP_BUYIN[fixed_map_name]     # buy-in is read from RemoteConfig (WageringSystem.maps.levelMilitary)

# -----------------------------
# 9) MAIN LOOP (150 DAYS)
# -----------------------------
for _day in range(days):
    # Run today's races (3 or 4).
    for _ in range(races_today()):
        for n in player_archetypes:
            # Skip if the player can't afford to enter.
            if wallets[n] < buy_in:
                continue

            # Pay entry fee.
            wallets[n] -= buy_in

            # Simulate finishing place for this archetype vs sampled opponents.
            place = placement_coupled(n)

            # Pay prize for podium finishes (1st/2nd/3rd).
            if isinstance(place, int) and 1 <= place <= 3:
                wallets[n] += payout_from_rewards(fixed_map_name, place)

            # Add XP and pay any level-up gold immediately for each level crossed.
            xp_tot[n] += xp_per_game_avg
            if use_levelup_gold:
                prev_level = levels[n]
                new_level  = level_from_xp(xp_tot[n])
                if new_level > prev_level:
                    # Sum gold for every level crossed (handles multi-level jumps).
                    wallets[n] += sum(level_to_gold.get(L, 0) for L in range(prev_level + 1, new_level + 1))
                    levels[n] = new_level

    # End-of-day daily login bonus (applied once per day).
    for n in player_archetypes:
        wallets[n] += daily_bonus_gold

# -----------------------------
# 10) SHOW RESULTS
# -----------------------------
summary = pd.DataFrame({
    "archetype": player_archetypes,
    "wallet":    [wallets[n] for n in player_archetypes],
    "level":     [levels[n]  for n in player_archetypes],
    "xp":        [int(xp_tot[n]) for n in player_archetypes],
})
print(summary)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
  archetype  wallet  level     xp
0   Skilled   47930     39  44363
1   Average   35850     39  44363
2    Novice    9020     38  41666


ISHA

In [ ]:
# ==============================================================
# Purpose: Simulate a 150-day economy for 6-player race lobbies,
#          estimating wallet growth, progression (XP/level), and
#          map selection under EV + epsilon-greedy behavior.
#
# How it works (high level):
# - Loads economy knobs (buy-ins, payouts, level XP, gold rewards) from remote CSV.
# - Samples race outcomes via coupled Gaussian skill (per archetype) + opponent mix.
# - Estimates per-map EV for each archetype, then picks maps epsilon-greedily (affordability-aware).
# - Updates wallet with buy-ins/payouts + daily bonus; grants level-up gold; accrues XP.
#
# Key inputs (easily tweakable): initial_wallet, daily_bonus_gold, days, EPSILON,
#   player_archetypes, NUM_PLAYERS, opponent_mix_weights, coupled_perf_mu/sigma.
#
# Outputs: final table (wallet, level, XP by archetype).
# (Extend to plot: wallet over time, time-to-unlock maps, races vs wallet.)
# ==============================================================


from google.colab import drive
import pandas as pd
import json
import random
import math
from bisect import bisect_right

# -----------------------------
# 1) LOAD REMOTE CONFIG (CSV)
# -----------------------------
drive.mount('/content/drive')

CONFIG_PATH = "/content/drive/MyDrive/OPT Volunteer Job Personal Folder/Week 5/remote config.csv"

df = pd.read_csv(CONFIG_PATH)
config = {}
for _, row in df.iterrows():
    k, t, v = row["key"], row["type"], row["value"]
    if t == "json":
        config[k] = json.loads(v)
    elif t == "int":
        config[k] = int(v)
    else:
        config[k] = v

# -----------------------------
# 2) READ KNOBS FROM CONFIG
# -----------------------------
initial_wallet = config["IAPstore"]["NewPlayerBonus"]      # e.g., 1200
daily_bonus_gold = 150
use_levelup_gold = True
days = 150
NUM_PLAYERS = 6
player_archetypes = ["Skilled", "Average", "Novice"]

# -----------------------------
# 3) RACE CADENCE
# -----------------------------
def races_today() -> int:
    return 3 + (1 if random.random() < 0.5 else 0)

# -----------------------------
# 4) PERFORMANCE MODEL
# -----------------------------
coupled_perf_mu = {"Skilled": 1.2, "Average": 1.0, "Novice": 0.6}
coupled_perf_sigma = 0.35
opponent_mix_weights = {"Skilled": 0.25, "Average": 0.50, "Novice": 0.25}

# -----------------------------
# 5) PROGRESSION (XP & LEVELS)
# -----------------------------
xp_per_game_avg = 81.7
cum_xp_by_level = config["LevelConfig"]["level_config"]
cum_list = cum_xp_by_level[:]
max_level = len(cum_list)

def level_from_xp(xp_total: float) -> int:
    idx = bisect_right(cum_list, xp_total) - 1
    return max(1, min(idx + 1, max_level))

level_to_gold = {i + 1: g for i, g in enumerate(config["GoldRewards"]["gold_rewards"])}

# -----------------------------
# 6) MAP DATA
# -----------------------------
name_map = {
    "levelMilitary": "Weapons Factory",
    "levelLA": "Lost Angeles",
    "levelElectric": "Electric Downtown",
    "levelMushroom": "Biodome",
    "levelWhitecity": "White City",
    "levelChinese": "Chinese",
}
maps_cfg = config["WageringSystem"]["maps"]

MAP_REWARDS = {name_map[k]: v["rewards"] for k, v in maps_cfg.items()}
MAP_BUYIN   = {name_map[k]: v["buy_in"]  for k, v in maps_cfg.items()}
ALL_MAPS = list(MAP_BUYIN.keys())

def payout_from_rewards(map_name: str, place: int) -> int:
    rewards = MAP_REWARDS[map_name]
    return rewards[place - 1] if 1 <= place <= 3 else 0

# -----------------------------
# 7) RACE OUTCOME
# -----------------------------
def placement_coupled(focal_archetype: str):
    kinds, weights = zip(*opponent_mix_weights.items())
    opponents = [random.choices(kinds, weights=weights, k=1)[0] for _ in range(NUM_PLAYERS - 1)]
    focal_score = random.gauss(coupled_perf_mu[focal_archetype], coupled_perf_sigma)
    opp_scores  = [random.gauss(coupled_perf_mu[a], coupled_perf_sigma) for a in opponents]
    scores      = [focal_score] + opp_scores
    order = sorted(range(NUM_PLAYERS), key=lambda i: scores[i], reverse=True)
    rank = order.index(0) + 1
    return rank if rank <= 3 else "other"

# -----------------------------
# 8) EV ESTIMATION
# -----------------------------
def estimate_ev_table(maps, N=1500):
    """
    Returns dict: EV_TABLE[archetype][map] = (EV, SD)
    """
    table = {arch: {} for arch in player_archetypes}
    for arch in player_archetypes:
        for m in maps:
            buyin = MAP_BUYIN[m]
            results = []
            for _ in range(N):
                place = placement_coupled(arch)
                payout = payout_from_rewards(m, place) if isinstance(place, int) else 0
                net = payout - buyin
                results.append(net)
            mean_ev = sum(results) / N
            variance = sum((x - mean_ev) ** 2 for x in results) / N
            sd = math.sqrt(variance)
            table[arch][m] = (mean_ev, sd)
    return table

# -----------------------------
# 9) MAP SELECTION (EPS-GREEDY)
# -----------------------------
EPSILON = 0.2

def choose_map_epsilon_greedy(archetype, wallet, maps, EV_TABLE, eps=EPSILON):
    affordable = [m for m in maps if wallet >= MAP_BUYIN[m]]
    if not affordable:
        return None
    if random.random() < eps:
        return random.choice(affordable)
    else:
        # Choose best EV among affordable maps
        best_map = max(affordable, key=lambda m: EV_TABLE[archetype][m][0])
        return best_map

# -----------------------------
# 10) SIMULATION STATE
# -----------------------------
wallets = {n: initial_wallet for n in player_archetypes}
xp_tot  = {n: 0.0 for n in player_archetypes}
levels  = {n: 1   for n in player_archetypes}

# Precompute EV table
EV_TABLE = estimate_ev_table(ALL_MAPS, N=1500)

# -----------------------------
# 11) MAIN LOOP (150 DAYS)
# -----------------------------
for _day in range(days):
    for _ in range(races_today()):
        for n in player_archetypes:
            chosen_map = choose_map_epsilon_greedy(n, wallets[n], ALL_MAPS, EV_TABLE, EPSILON)
            if not chosen_map:
                continue
            buy_in = MAP_BUYIN[chosen_map]
            wallets[n] -= buy_in
            place = placement_coupled(n)
            if isinstance(place, int) and 1 <= place <= 3:
                wallets[n] += payout_from_rewards(chosen_map, place)
            xp_tot[n] += xp_per_game_avg
            if use_levelup_gold:
                prev_level = levels[n]
                new_level  = level_from_xp(xp_tot[n])
                if new_level > prev_level:
                    wallets[n] += sum(level_to_gold.get(L, 0) for L in range(prev_level + 1, new_level + 1))
                    levels[n] = new_level
    for n in player_archetypes:
        wallets[n] += daily_bonus_gold

# -----------------------------
# 12) SHOW RESULTS
# -----------------------------
summary = pd.DataFrame({
    "archetype": player_archetypes,
    "wallet":    [wallets[n] for n in player_archetypes],
    "level":     [levels[n]  for n in player_archetypes],
    "xp":        [int(xp_tot[n]) for n in player_archetypes],
})
print(summary)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
  archetype  wallet  level     xp
0   Skilled    2920     38  43546
1   Average    1560     38  43382
2    Novice    1010     33  28431


**Below Code is Owned By Whom? **

In [ ]:
# ==============================================================
# Purpose: Simulate a 150-day economy for 6-player race lobbies,
#          estimating wallet growth, progression (XP/level), and
#          map selection under EV + epsilon-greedy behavior.
#
# Modifications:
# - Weaker archetypes have higher variance in skill (less consistent).
# - EV summary now shows both expected value (EV) and standard deviation (SD).
# ==============================================================

from google.colab import drive
import pandas as pd
import json
import random
import math
from bisect import bisect_right

# -----------------------------
# 1) LOAD REMOTE CONFIG (CSV)
# -----------------------------
drive.mount('/content/drive')

CONFIG_PATH = "/content/drive/MyDrive/OPT Volunteer Job Personal Folder/Week 5/remote config.csv"

df = pd.read_csv(CONFIG_PATH)
config = {}
for _, row in df.iterrows():
    k, t, v = row["key"], row["type"], row["value"]
    if t == "json":
        config[k] = json.loads(v)
    elif t == "int":
        config[k] = int(v)
    else:
        config[k] = v

# -----------------------------
# 2) READ KNOBS FROM CONFIG
# -----------------------------
initial_wallet = config["IAPstore"]["NewPlayerBonus"]      # e.g., 1200
daily_bonus_gold = 150
use_levelup_gold = True
days = 150
NUM_PLAYERS = 6
player_archetypes = ["Skilled", "Average", "Novice"]

# -----------------------------
# 3) RACE CADENCE
# -----------------------------
def races_today() -> int:
    return 3 + (1 if random.random() < 0.5 else 0)

# -----------------------------
# 4) PERFORMANCE MODEL
# -----------------------------
# Mean skill levels
coupled_perf_mu = {
    "Skilled": 1.2,
    "Average": 1.0,
    "Novice":  0.6
}

# Std dev of performance (weaker = higher variance)
coupled_perf_sigma = {
    "Skilled": 0.15,
    "Average": 0.25,
    "Novice":  0.350
}

# Opponent mix distribution
opponent_mix_weights = {
    "Skilled": 0.25,
    "Average": 0.50,
    "Novice":  0.25
}

# -----------------------------
# 5) PROGRESSION (XP & LEVELS)
# -----------------------------
xp_per_game_avg = 81.7
cum_xp_by_level = config["LevelConfig"]["level_config"]
cum_list = cum_xp_by_level[:]
max_level = len(cum_list)

def level_from_xp(xp_total: float) -> int:
    idx = bisect_right(cum_list, xp_total) - 1
    return max(1, min(idx + 1, max_level))

level_to_gold = {i + 1: g for i, g in enumerate(config["GoldRewards"]["gold_rewards"])}

# -----------------------------
# 6) MAP DATA
# -----------------------------
name_map = {
    "levelMilitary": "Weapons Factory",
    "levelLA": "Lost Angeles",
    "levelElectric": "Electric Downtown",
    "levelMushroom": "Biodome",
    "levelWhitecity": "White City",
    "levelChinese": "Chinese",
}
maps_cfg = config["WageringSystem"]["maps"]

MAP_REWARDS = {name_map[k]: v["rewards"] for k, v in maps_cfg.items()}
MAP_BUYIN   = {name_map[k]: v["buy_in"]  for k, v in maps_cfg.items()}
ALL_MAPS = list(MAP_BUYIN.keys())

def payout_from_rewards(map_name: str, place: int) -> int:
    rewards = MAP_REWARDS[map_name]
    return rewards[place - 1] if 1 <= place <= 3 else 0

# -----------------------------
# 7) RACE OUTCOME
# -----------------------------
def placement_coupled(focal_archetype: str):
    # Sample opponent archetypes
    kinds, weights = zip(*opponent_mix_weights.items())
    opponents = [random.choices(kinds, weights=weights, k=1)[0] for _ in range(NUM_PLAYERS - 1)]

    # Focal player
    focal_score = random.gauss(
        coupled_perf_mu[focal_archetype],
        coupled_perf_sigma[focal_archetype]
    )

    # Opponents
    opp_scores = [
        random.gauss(coupled_perf_mu[a], coupled_perf_sigma[a])
        for a in opponents
    ]

    scores = [focal_score] + opp_scores
    order = sorted(range(NUM_PLAYERS), key=lambda i: scores[i], reverse=True)
    rank = order.index(0) + 1
    return rank if rank <= 3 else "other"

# -----------------------------
# 8) EV ESTIMATION
# -----------------------------
def estimate_ev_table(maps, N=1500):
    """
    Returns dict: EV_TABLE[archetype][map] = (EV, SD, CI_low, CI_high)
    """
    table = {arch: {} for arch in player_archetypes}
    for arch in player_archetypes:
        for m in maps:
            buyin = MAP_BUYIN[m]
            results = []
            for _ in range(N):
                place = placement_coupled(arch)
                payout = payout_from_rewards(m, place) if isinstance(place, int) else 0
                net = payout - buyin
                results.append(net)
            mean_ev = sum(results) / N
            variance = sum((x - mean_ev) ** 2 for x in results) / N
            sd = math.sqrt(variance)
            se = sd / math.sqrt(N)  # standard error
            ci_low = mean_ev - 1.96 * se
            ci_high = mean_ev + 1.96 * se
            table[arch][m] = (mean_ev, sd, ci_low, ci_high)
    return table
# -----------------------------
# 9) MAP SELECTION (EPS-GREEDY)
# -----------------------------
EPSILON = 0.2

def choose_map_epsilon_greedy(archetype, wallet, maps, EV_TABLE, eps=EPSILON):
    affordable = [m for m in maps if wallet >= MAP_BUYIN[m]]
    if not affordable:
        return None
    if random.random() < eps:
        return random.choice(affordable)
    else:
        # Choose best EV among affordable maps
        best_map = max(affordable, key=lambda m: EV_TABLE[archetype][m][0])
        return best_map

# -----------------------------
# 10) SIMULATION STATE
# -----------------------------
wallets = {n: initial_wallet for n in player_archetypes}
xp_tot  = {n: 0.0 for n in player_archetypes}
levels  = {n: 1   for n in player_archetypes}

# Precompute EV table
EV_TABLE = estimate_ev_table(ALL_MAPS, N=1500)

# -----------------------------
# 11) MAIN LOOP (150 DAYS)
# -----------------------------
for _day in range(days):
    for _ in range(races_today()):
        for n in player_archetypes:
            chosen_map = choose_map_epsilon_greedy(n, wallets[n], ALL_MAPS, EV_TABLE, EPSILON)
            if not chosen_map:
                continue
            buy_in = MAP_BUYIN[chosen_map]
            wallets[n] -= buy_in
            place = placement_coupled(n)
            if isinstance(place, int) and 1 <= place <= 3:
                wallets[n] += payout_from_rewards(chosen_map, place)
            xp_tot[n] += xp_per_game_avg
            if use_levelup_gold:
                prev_level = levels[n]
                new_level  = level_from_xp(xp_tot[n])
                if new_level > prev_level:
                    wallets[n] += sum(level_to_gold.get(L, 0) for L in range(prev_level + 1, new_level + 1))
                    levels[n] = new_level
    for n in player_archetypes:
        wallets[n] += daily_bonus_gold
# -----------------------------
# 12) SHOW RESULTS
# -----------------------------
summary = pd.DataFrame({
    "archetype": player_archetypes,
    "wallet":    [wallets[n] for n in player_archetypes],
    "level":     [levels[n]  for n in player_archetypes],
    "xp":        [int(xp_tot[n]) for n in player_archetypes],
})
print("\n=== Final 150-Day Summary ===")
print(summary)

# Show EV + SD + CI table for transparency
rows = []
for arch in player_archetypes:
    for m in ALL_MAPS:
        ev, sd, ci_low, ci_high = EV_TABLE[arch][m]
        rows.append([arch, m, round(ev, 2), round(sd, 2), round(ci_low, 2), round(ci_high, 2)])
ev_df = pd.DataFrame(rows, columns=["Archetype", "Map", "EV", "SD", "95% CI Low", "95% CI High"])
print("\n=== EV Table (mean net return ± variability, with 95% CI) ===")
print(ev_df)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).

=== Final 150-Day Summary ===
  archetype  wallet  level     xp
0   Skilled   43950     38  42729
1   Average    5360     38  42647
2    Novice     280     33  29166

=== EV Table (mean net return ± variability, with 95% CI) ===
   Archetype                Map          EV         SD  95% CI Low  \
0    Skilled    Weapons Factory      -33.69      59.11      -36.68   
1    Skilled       Lost Angeles     -222.20     691.41     -257.19   
2    Skilled  Electric Downtown     -961.20    2790.91    -1102.44   
3    Skilled            Biodome    -9000.00   28315.30   -10432.95   
4    Skilled         White City   -46140.00   55901.81   -48969.02   
5    Skilled            Chinese  -305025.00  646079.71  -337721.14   
6    Average    Weapons Factory      -80.99      69.10      -84.48   
7    Average       Lost Angeles     -763.53     781.19     -803.07   
8    Averag